# 1. Setup

**Approach 2 · A2 with CLIP disease evidence and adaptive calibrated stacking**

Open this notebook inside the team repository in VS Code. Install the project dependencies and use **Run All**. `DEBUG=False` runs the full study. `DEBUG=True` checks the encoders and training on a small train-only subset. Frozen DINOv2 and OpenCLIP weights download on first use.

The historical A2 result was 63.90% seen / 9.39% unseen Top-1 with raw scoring and 52.10% / 63.33% with static calibration. This study reruns B0 and A2 side by side and measures both raw transfer and seen/unseen balance. The target is an absolute Top-1 gap of at most 10 percentage points with a seen loss of at most 2 points relative to the corresponding control, plus positive gains in unseen Top-1, unseen Top-5 and harmonic mean.

In [ ]:
DEBUG = False

from datetime import datetime, timezone
from pathlib import Path
import copy
from dataclasses import dataclass
import hashlib
import json
import math
import sys

import open_clip
import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_crop_and_disease_names, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 0
BATCH_SIZE, EPOCHS = (8, 1) if DEBUG else (16, 20)
HEAD_LR, WEIGHT_DECAY = 1e-4, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.10, 0.10
CLIP_MODEL = "ViT-B-32-quickgelu"
CLIP_WEIGHTS = "openai"
CLIP_SCALE, MARGIN_SCALE = 16.0, 1.0
CROP_PROMPTS = ("a photo of a {} plant leaf", "a close-up photo of foliage on a {} plant")
DISEASE_PROMPTS = ("a close-up photo of a plant leaf with {}", "a photo of {} symptoms on a plant leaf")
HEALTHY_PROMPTS = ("a close-up photo of a healthy plant leaf", "a photo of a plant leaf without symptoms")
BLENDS = (0.0, 0.25, 0.5, 0.75, 1.0)
GAMMAS = [value / 2 for value in range(25)]
ADAPTIVE_WEIGHTS = (0., 1., 2.)
SEEN_DROP_LIMIT, GAP_TARGET = 2.0, 10.0
MODEL_NAME = "approach2_a2_clip_balance"
RUN_NAME = MODEL_NAME + ("_debug" if DEBUG else "")
B0_UNSEEN_REFERENCE = 9.60
RAW_UNSEEN_TARGET = round(B0_UNSEEN_REFERENCE * 1.30, 2)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULTS_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
DEVELOPMENT_DIR = RESULTS_DIR / "development"
OFFICIAL_DIR = RESULTS_DIR / "official" / "pair_identity_order"
CHECKPOINT_DIR = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID

SEMANTIC_CHECKPOINT = CHECKPOINT_DIR / "clip_semantics.pt"

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError("Training CSV or image directory is missing.")
if (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) != (323, 10, 333):
    raise ValueError("Use the shared 323/333 evaluation API.")
print(f"PyTorch {torch.__version__} · {DEVICE} · debug={DEBUG} · run {RUN_ID}")


# 2. Data

Read only `train.csv` for development. Remove every image of ten training-derived combinations from gradient updates, while keeping each component class represented in fit. Tune on five of these proxy combinations and a seen validation subset. Reserve the other five proxy combinations and disjoint seen validation images for a check after selection. Development uses the same 323 training-derived candidate pairs throughout. Full refit restores all training rows.

In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


def proxy_split(fit_frame, val_frame):
    """Hold out complete training pairs from the development heads only."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Proxy validation needs seen and held-out images.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
_, image_transform = get_transforms("imagenet")

settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "head_lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW", "schedule": "CosineAnnealingLR", "label_smoothing": SMOOTHING,
    "backbone_frozen": True, "decor_weight": DECOR_WEIGHT, "supcon_weight": SUPCON_WEIGHT,
    "temperature": TEMPERATURE,
    "semantic_encoder": {"architecture": CLIP_MODEL, "weights": CLIP_WEIGHTS, "scale": CLIP_SCALE,
                         "crop_prompts": list(CROP_PROMPTS), "disease_prompts": list(DISEASE_PROMPTS),
                         "healthy_prompts": list(HEALTHY_PROMPTS)},
    "rule_grid": {"blend": list(BLENDS), "gamma": GAMMAS, "adaptive": list(ADAPTIVE_WEIGHTS)},
    "margin_scale": MARGIN_SCALE, "seen_drop_limit_pp": SEEN_DROP_LIMIT, "gap_target_pp": GAP_TARGET,
    "gamma_grid": GAMMAS, "gamma_selection": "proxy harmonic mean, seen top1, lower gamma",
    "train_csv_sha256": sha256_file(team.TRAIN_CSV),
    "torch": str(torch.__version__), "timm": timm.__version__, "debug": DEBUG,
}


@dataclass(frozen=True)
class Trial:
    code: str
    development_dir: Path
    final_dir: Path
    checkpoint: Path
    lock: Path
    settings: dict


def make_trial(code):
    final_dir = RESULTS_DIR / "final" / code
    return Trial(code, DEVELOPMENT_DIR / code, final_dir, CHECKPOINT_DIR / code / "model.pt",
                 final_dir / "selection_lock.json",
                 {**settings, "use_approach2_losses": code == "A2",
                  "epoch_selection": "shared_epoch_from_B0_raw_proxy_harmonic_mean"})


baseline, approach2 = make_trial("B0"), make_trial("A2")
trials = (approach2,) if DEBUG else (baseline, approach2)


# 3. Model

- **B0** uses frozen DINOv2 ViT-B/14, two linear concept heads and cross-entropy.
- **A2** adds the uploaded notebook's crop–disease logit Decorrelation and Cross-crop SupCon. Both models get the same team image augmentation.
- **Fusion raw** blends the A2 disease probability with CLIP disease probability, using a crop-independent prompt bank built solely from training disease names. The A2 crop head is retained. The pair score is the sum of crop and blended disease log-probabilities for every candidate. The seen-pair penalty is zero.
- **Balanced** adds adaptive Calibrated Stacking. A separate CLIP crop/disease score supplies a margin between the best novel and best seen candidates. For image $x$, subtract $\max(0, \gamma + \beta \tanh((m_{novel}(x)-m_{seen}(x))/T))$ from seen scores. Larger independent evidence for a novel pair increases the penalty. The rule with $\beta=0$ gives ordinary Calibrated Stacking.

References. [CLIP, ICML 2021](https://proceedings.mlr.press/v139/radford21a.html) · [Supervised Contrastive Learning, NeurIPS 2020](https://proceedings.neurips.cc/paper_files/paper/2020/hash/d89a66c7c80a29b1bdbab0f2a1a94af8-Abstract.html) · [Calibrated Stacking, ECCV 2016](https://arxiv.org/abs/1605.04253)

In [ ]:
class MultiTaskNet(nn.Module):
    """Latest Baseline 2 architecture."""
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.crop_head = nn.Linear(self.backbone.num_features, NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


def pair_scores(crop_logits, disease_logits, pairs):
    pairs = pairs.to(crop_logits.device)
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()


def concept_text(name):
    return " ".join(str(name).replace("_", " ").split())


@torch.inference_mode()
def make_semantic_encoder():
    model = open_clip.create_model(CLIP_MODEL, pretrained=CLIP_WEIGHTS).to(DEVICE).eval()
    tokenizer = open_clip.get_tokenizer(CLIP_MODEL)
    crops, diseases = get_crop_and_disease_names()

    def encode_bank(names, count, prompts, healthy_prompts=None):
        if set(names) != set(range(count)):
            raise ValueError("Training metadata must name each crop and disease.")
        texts = []
        for index in range(count):
            name = concept_text(names[index])
            templates = healthy_prompts if healthy_prompts and name.casefold() == "healthy" else prompts
            texts.extend(template.format(name) for template in templates)
        tokens = tokenizer(texts)
        embeddings = torch.cat([model.encode_text(batch.to(DEVICE), normalize=True) for batch in tokens.split(32)])
        return F.normalize(embeddings.float().reshape(count, len(prompts), -1).mean(1), dim=1)

    bank = {"crop": encode_bank(crops, NUM_CROPS, CROP_PROMPTS),
            "disease": encode_bank(diseases, NUM_DISEASES, DISEASE_PROMPTS, HEALTHY_PROMPTS)}
    return model, bank


@torch.inference_mode()
def semantic_log_probs(model, bank, images):
    # The common loader uses the same 224px pixels for both encoders.
    image_mean, image_std = (images.new_tensor(v).view(1, 3, 1, 1) for v in team.NORM_STATS["imagenet"])
    clip_mean, clip_std = (images.new_tensor(v).view(1, 3, 1, 1) for v in team.NORM_STATS["clip"])
    clip_input = (images * image_std + image_mean - clip_mean) / clip_std
    features = model.encode_image(clip_input, normalize=True).float()
    return tuple((CLIP_SCALE * (features @ bank[key].T)).log_softmax(1) for key in ("crop", "disease"))


def combined_pair_scores(crop, disease, clip_crop, clip_disease, pairs, blend):
    dp = disease_log_probs(disease, clip_disease, blend)
    return crop.float().log_softmax(1)[:, pairs[:, 0]] + dp[:, pairs[:, 1]]


def disease_log_probs(disease, clip_disease, blend):
    dp = disease.float().log_softmax(1)
    if blend == 1.0:
        dp = clip_disease
    elif blend:
        dp = torch.logaddexp(dp + math.log1p(-blend), clip_disease + math.log(blend))
    return dp


def clip_pair_scores(crop_log_probs, disease_log_probs, pairs):
    return crop_log_probs[:, pairs[:, 0]] + disease_log_probs[:, pairs[:, 1]]


def semantic_margin(scores, membership):
    best_seen = scores[:, membership].amax(dim=1)
    best_new = scores[:, ~membership].amax(dim=1)
    return ((best_new - best_seen) / MARGIN_SCALE).tanh()


def apply_rule(scores, membership, margin, gamma, adaptive):
    penalty = (gamma + adaptive * margin).clamp_min(0)
    return scores - penalty[:, None] * membership


# 4. Training

Choose a common epoch by B0's raw tune harmonic mean. At that epoch, select each control's static gamma on tune. Select the fusion blend and the adaptive rule using the corresponding A2 control. Rank rules by the seen loss limit first, then improvement across all three metrics and the balance target. Identify the rule that maximizes the smallest metric gain when several meet the same criteria. The check split reports how the chosen rules transfer to new proxy pairs. It does not change the rules.

Refit fresh heads for the locked epoch count on all training images. Save complete DINOv2 checkpoints for B0 and A2 and a separate CLIP visual encoder plus its frozen training-derived text bank. Lock the parameters and hashes before official test access.

In [ ]:
def selection_key(row):
    return (row["harmonic_top1"], row["proxy_top1"], row["seen_top1"], -row["epoch"])


def snapshot_tail(tail):
    return {name: value.detach().cpu().clone() for name, value in tail.state_dict().items()}


def read_final_lock(trial):
    if not trial.lock.is_file() or not trial.checkpoint.is_file():
        raise FileNotFoundError("Complete full refit first.")
    saved = json.loads(trial.lock.read_text(encoding="utf-8"))
    if saved["settings"] != trial.settings or saved["checkpoint_sha256"] != sha256_file(trial.checkpoint):
        raise RuntimeError("Checkpoint or settings changed after selection.")
    if saved["seen_pairs"] != train_pairs.tolist():
        raise RuntimeError("Training candidate identities changed after selection.")
    return saved


METRICS = ("proxy_top1", "proxy_top5", "harmonic_top1")
OFFICIAL_METRICS = ("unseen_top1", "unseen_top5", "harmonic_mean")


def metrics(seen_group, proxy_group, membership, gamma=0., adaptive=0.):
    seen_s, seen_y, seen_margin = seen_group
    proxy_s, proxy_y, proxy_margin = proxy_group
    seen_s = apply_rule(seen_s, membership, seen_margin, gamma, adaptive)
    proxy_s = apply_rule(proxy_s, membership, proxy_margin, gamma, adaptive)
    seen = top_k_accuracy(seen_s, seen_y, 1)
    unseen = top_k_accuracy(proxy_s, proxy_y, 1)
    return {"gamma": gamma, "adaptive": adaptive, "seen_top1": seen, "proxy_top1": unseen,
            "proxy_top5": top_k_accuracy(proxy_s, proxy_y, 5), "harmonic_top1": harmonic_mean(seen, unseen),
            "gap_pp": round(abs(seen - unseen), 2)}


def gains(result, ref, keys):
    return [round(result[k] - ref[k], 2) for k in keys]


def candidate_key(result, ref):
    delta = gains(result, ref, METRICS)
    seen_ok = result["seen_top1"] >= ref["seen_top1"] - SEEN_DROP_LIMIT
    joint = min(delta) > 0
    return (seen_ok, joint, joint and result["gap_pp"] <= GAP_TARGET, min(delta), sum(delta),
            -result["gap_pp"], -result["blend"], -result["adaptive"], -result["gamma"])


def select_rules(units, tune, check, fit_mask, best, frames):
    for code, heads in units.items():
        heads.load_state_dict(best[code]["weights"])

    def groups(code, blend, cached):
        return [score_cache(units[code], view, fit_mask, blend) for view in cached]

    reference = {}
    rows = []
    for code in units:
        raw_groups = groups(code, 0., tune)
        grid = [{"recipe": code, "blend": 0., **metrics(*raw_groups, fit_mask, gamma)} for gamma in GAMMAS]
        rows.extend(grid)
        reference[code] = {"raw": grid[0],
                           "calibrated": max(grid, key=lambda row: (row["harmonic_top1"], row["seen_top1"],
                                                                   -row["gamma"]))}
    candidates = []
    for blend in BLENDS:
        fused = groups("A2", blend, tune)
        for gamma in GAMMAS:
            for beta in ADAPTIVE_WEIGHTS:
                candidates.append({"recipe": "A2", "blend": blend, **metrics(*fused, fit_mask, gamma, beta)})

    raw = max((row for row in candidates if row["gamma"] == row["adaptive"] == 0),
              key=lambda row: candidate_key(row, reference["A2"]["raw"]))
    balanced = max(candidates, key=lambda row: candidate_key(row, reference["A2"]["calibrated"]))

    rules = {}
    for code, ref in reference.items():
        rules[code] = {f"{code}_{name}": {k: row[k] for k in ("blend", "gamma", "adaptive")}
                       for name, row in ref.items()}
    rules["A2"].update({label: {k: row[k] for k in ("blend", "gamma", "adaptive")}
                        for label, row in (("Fusion_raw", raw), ("Balanced", balanced))})

    pd.DataFrame(rows).to_csv(DEVELOPMENT_DIR / "reference_tune_grid.csv", index=False)
    pd.DataFrame(candidates).to_csv(DEVELOPMENT_DIR / "fusion_tune_grid.csv", index=False)
    check_rows = []
    for code, rule_group in rules.items():
        for name, rule in rule_group.items():
            check_rows.append({"model": name, "recipe": code, "blend": rule["blend"],
                               **metrics(*groups(code, rule["blend"], check), fit_mask,
                                         rule["gamma"], rule["adaptive"])})
    pd.DataFrame(check_rows).to_csv(DEVELOPMENT_DIR / "independent_proxy_check.csv", index=False)
    checks = comparisons(check_rows, METRICS)
    pd.DataFrame(checks).to_csv(DEVELOPMENT_DIR / "independent_proxy_gains.csv", index=False)
    display(pd.DataFrame(checks))
    return {"selected_epoch": best["B0"]["metrics"]["epoch"], "rules": rules,
            "tune_pairs": sorted(frames["tune_proxy"].pair_row.unique().tolist()),
            "check_pairs": sorted(frames["check_proxy"].pair_row.unique().tolist()),
            "selection": {code: item["metrics"] for code, item in best.items()},
            "independent_proxy_check": check_rows, "independent_proxy_gains": checks}


def comparisons(rows, keys):
    lookup = {row["model"]: row for row in rows}
    result = []
    for label, reference in (("Fusion_raw", "A2_raw"), ("Fusion_raw", "B0_raw"),
                             ("Balanced", "A2_calibrated"), ("Balanced", "B0_calibrated")):
        model, control = lookup[label], lookup[reference]
        delta = gains(model, control, keys)
        gap = round(abs(model["seen_top1"] - model[keys[0]]), 2)
        seen_change = round(model["seen_top1"] - control["seen_top1"], 2)
        result.append({"model": label, "reference": reference,
                       **{f"{key}_gain_pp": value for key, value in zip(keys, delta)},
                       "seen_top1_change_pp": seen_change, "seen_unseen_gap_pp": gap,
                       "all_three_increased": min(delta) > 0,
                       "all_three_and_balance": min(delta) > 0 and gap <= GAP_TARGET
                       and seen_change >= -SEEN_DROP_LIMIT})
    return result



def heads_from(model):
    return nn.ModuleDict({"crop": copy.deepcopy(model.crop_head), "disease": copy.deepcopy(model.disease_head)})


@torch.no_grad()
def cache_features(backbone, frame, semantic=None):
    loader = DataLoader(PlantDataset(frame, image_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, generator=torch.Generator().manual_seed(team.SEED))
    batches = []
    for images, _, _, labels in loader:
        images = images.to(DEVICE)
        features = backbone(images).cpu()
        clip = () if semantic is None else tuple(z.cpu() for z in semantic_log_probs(*semantic, images))
        batches.append((features, labels, *clip))
    return tuple(torch.cat(list(parts)) for parts in zip(*batches))


@torch.no_grad()
def score_cache(heads, cache, membership, blend=0.):
    features, labels, *clip = cache
    crop, disease = heads["crop"](features.to(DEVICE)).cpu(), heads["disease"](features.to(DEVICE)).cpu()
    if clip:
        scores = combined_pair_scores(crop, disease, *clip, train_pairs, blend)
        margin = semantic_margin(clip_pair_scores(*clip, train_pairs), membership)
    else:
        scores = pair_scores(crop, disease, train_pairs)
        margin = scores.new_zeros(len(labels))
    return scores, labels, margin


def fit_heads(model, loader, epochs, fit_mask=None, tune=None):
    units = {code: heads_from(model) for code in epochs}
    optimizers = {code: torch.optim.AdamW(heads.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
                  for code, heads in units.items()}
    schedules = {code: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
                 for code, opt in optimizers.items()}
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    history, best = [], {}

    for epoch in range(1, max(epochs.values()) + 1):
        active = [code for code, limit in epochs.items() if epoch <= limit]
        totals = {code: [0., 0] for code in active}
        examples = 0
        for step, (images, crop, disease, _) in enumerate(loader, 1):
            images, crop, disease = images.to(DEVICE), crop.to(DEVICE), disease.to(DEVICE)
            with torch.no_grad():
                features = model.backbone(images)
            examples += len(images)
            for code in active:
                heads, optimizer = units[code], optimizers[code]
                crop_logits, disease_logits = heads["crop"](features), heads["disease"](features)
                loss = criterion(crop_logits, crop) + criterion(disease_logits, disease)
                anchors = 0
                if code == "A2":
                    crop_repr, disease_repr = center_logits(crop_logits), center_logits(disease_logits)
                    supcon, anchors = cross_crop_supcon(disease_repr, crop, disease)
                    loss = loss + DECOR_WEIGHT * decorrelation(crop_repr, disease_repr) + SUPCON_WEIGHT * supcon
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite training loss for {code}.")
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                record = totals[code]
                record[0] += loss.item() * len(images)
                record[1] += int(anchors)
            if step == 1 or step % 50 == 0 or step == len(loader):
                print(f"Epoch {epoch}/{max(epochs.values())} · batch {step}/{len(loader)}", flush=True)
        if examples != len(loader.dataset):
            raise RuntimeError("Training did not process every fit image exactly once.")

        epoch_metrics = {}
        for code in active:
            schedules[code].step()
            total, anchors = totals[code]
            row = {"recipe": code, "epoch": epoch, "images_trained": examples, "train_loss": total / examples,
                   "supcon_anchor_fraction": anchors / examples}
            if tune is not None:
                report = metrics(*(score_cache(units[code], group, fit_mask) for group in tune), fit_mask)
                row.update({key: value for key, value in report.items() if key not in ("gamma", "adaptive")})
                epoch_metrics[code] = row
            history.append(row)
        if tune is not None and (
            not best or selection_key(epoch_metrics["B0"]) > selection_key(best["B0"]["metrics"])
        ):
            best = {code: {"metrics": metrics.copy(), "weights": snapshot_tail(units[code])}
                    for code, metrics in epoch_metrics.items()}
        print(pd.DataFrame(history[-len(active):]).to_string(index=False), flush=True)
    return units, history, best


def run_development():
    team.set_seed()
    train, val, seen, unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if seen is not None or unseen is not None:
        raise RuntimeError("Development loaders must use only the training CSV.")
    train.pin_memory = False
    model = MultiTaskNet(pretrained=True).to(DEVICE).eval()
    fit_mask, frames = (None, {}) if DEBUG else proxy_split(train.dataset.frame, val.dataset.frame)
    if fit_mask is not None:
        allowed = fit_mask.nonzero(as_tuple=True)[0].tolist()
        fit_rows = train.dataset.frame.loc[train.dataset.frame.pair_row.isin(allowed)]
        train.dataset.frame = fit_rows.reset_index(drop=True)
    tune = None if DEBUG else [cache_features(model.backbone, frames[key]) for key in ("tune_seen", "tune_proxy")]
    units, history, best = fit_heads(model, train, {trial.code: EPOCHS for trial in trials}, fit_mask, tune)
    for trial in trials:
        trial.development_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame([row for row in history if row["recipe"] == trial.code]).to_csv(
            trial.development_dir / "training_history.csv", index=False)
    semantic = make_semantic_encoder()
    if DEBUG:
        probe = cache_features(model.backbone, val.dataset.frame, semantic)
        if not all(torch.isfinite(values).all() for values in probe[2:]):
            raise ValueError("Non-finite CLIP debug scores.")
        print("Verified CLIP image and text encoders on training data.")
        return {}

    tune = [cache_features(model.backbone, frames[key], semantic) for key in ("tune_seen", "tune_proxy")]
    check = [cache_features(model.backbone, frames[key], semantic) for key in ("check_seen", "check_proxy")]
    plan = select_rules(units, tune, check, fit_mask, best, frames)
    SEMANTIC_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    clip_model, bank = semantic
    atomic_save(SEMANTIC_CHECKPOINT, {"visual": clip_model.visual.cpu().state_dict(),
                                     "bank": {key: value.cpu() for key, value in bank.items()}})
    plan["clip_sha256"] = sha256_file(SEMANTIC_CHECKPOINT)
    atomic_json(DEVELOPMENT_DIR / "research_plan.json", plan)
    return plan


plan = run_development()


In [ ]:
def refit_full_train():
    if any(trial.lock.is_file() for trial in trials):
        raise FileExistsError("This run is already locked. Rerun Setup for a new run.")
    if sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]:
        raise RuntimeError("Training CSV changed after development.")
    frozen_plan = json.loads((DEVELOPMENT_DIR / "research_plan.json").read_text(encoding="utf-8"))
    if frozen_plan != plan:
        raise RuntimeError("Training-only research plan changed.")
    epochs = {trial.code: plan["selected_epoch"] for trial in trials}
    if not 1 <= plan["selected_epoch"] <= EPOCHS:
        raise ValueError("Invalid selected epoch count.")

    team.set_seed()
    model = MultiTaskNet(pretrained=True).to(DEVICE).eval()
    train_transform, _ = get_transforms("imagenet")
    loader = DataLoader(PlantDataset(train_full, train_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=0, drop_last=False,
                        generator=torch.Generator().manual_seed(team.SEED))
    units, history, _ = fit_heads(model, loader, epochs)
    runs = {}
    model.cpu()
    for trial in trials:
        code = trial.code
        model.crop_head.load_state_dict(units[code]["crop"].state_dict())
        model.disease_head.load_state_dict(units[code]["disease"].state_dict())
        rules = plan["rules"][code]
        refit = {"initialization": "original_pretrained_dinov2_fresh_heads", "data": "all_training_csv_rows",
                 "images_per_epoch": len(train_full), "training_pairs": len(train_pairs),
                 "epochs": epochs[code], "cosine_t_max": EPOCHS}
        metadata = {"settings": trial.settings, "recipe": code, "refit": refit,
                    "seen_pairs": train_pairs.tolist(), "selection": plan["selection"][code],
                    "prediction_rules": rules}
        trial.final_dir.mkdir(parents=True, exist_ok=True)
        trial.checkpoint.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame([row for row in history if row["recipe"] == code]).to_csv(
            trial.final_dir / "refit_history.csv", index=False)
        atomic_save(trial.checkpoint, {"state_dict": model.state_dict(), **metadata})
        locked = {**metadata, "research_plan": plan, "checkpoint_sha256": sha256_file(trial.checkpoint),
                  "checkpoint_bytes": trial.checkpoint.stat().st_size}
        atomic_json(trial.lock, locked)
        runs[code] = locked
    return runs


runs = {} if DEBUG else refit_full_train()


# 5. Evaluation

Check every frozen artifact before opening official files. Combine 323 train-derived pairs and ten distinct unseen identities read from the crop and disease columns only. Use filenames separately for inference. Build every Top-1 and Top-5 prediction for both models and all rules before joining per-image labels. Report seen, unseen and overall accuracy against all 333 candidates. Restricted-candidate and true-crop scores are diagnostic. Official results never change a model or threshold.

In [ ]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, semantic, blends, perturb=None):
    generator = torch.Generator().manual_seed(team.SEED)
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, generator=generator)
    mean, std = (torch.tensor(values).view(1, 3, 1, 1) for values in team.NORM_STATS["imagenet"])
    pair_indices = pairs.to(DEVICE)
    scores, concepts, margins = {blend: [] for blend in blends}, {blend: [] for blend in blends}, []
    membership = torch.arange(len(pairs), device=DEVICE) < len(train_pairs)
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        features = model.backbone(images.to(DEVICE))
        crop, disease = model.crop_head(features), model.disease_head(features)
        clip = None if semantic is None else semantic_log_probs(*semantic, images.to(DEVICE))
        margin = (crop.new_zeros(len(crop)) if clip is None
                  else semantic_margin(clip_pair_scores(*clip, pair_indices), membership))
        margins.append(margin.cpu())
        for blend in blends:
            batch = (pair_scores(crop, disease, pair_indices) if clip is None
                     else combined_pair_scores(crop, disease, *clip, pair_indices, blend)).cpu()
            if batch.shape != (len(images), len(pairs)) or not torch.isfinite(batch).all():
                raise ValueError("Invalid raw pair scores.")
            scores[blend].append(batch)
            disease_evidence = disease if not blend else disease_log_probs(disease, clip[1], blend)
            concepts[blend].append(torch.stack((crop.argmax(1), disease_evidence.argmax(1)), dim=1).cpu())
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return ({blend: torch.cat(batches) for blend, batches in scores.items()},
            {blend: torch.cat(batches) for blend, batches in concepts.items()}, torch.cat(margins))


def predict_trial(trial, run, pairs, seen_names, unseen_names, semantic):
    rules = run["prediction_rules"]
    if rules != plan["rules"][trial.code] or run["research_plan"] != plan:
        raise ValueError("Final inference rules differ from the training-only lock.")
    if any(rule["gamma"] not in GAMMAS or rule["blend"] not in BLENDS
           or rule["adaptive"] not in ADAPTIVE_WEIGHTS for rule in rules.values()):
        raise ValueError("Out-of-grid inference rule.")
    saved = torch.load(trial.checkpoint, map_location="cpu", weights_only=True)
    for key in ("settings", "prediction_rules", "seen_pairs", "selection", "refit"):
        if saved[key] != run[key]:
            raise ValueError(f"Checkpoint and selection lock differ for {key}.")
    model = MultiTaskNet(pretrained=False)
    model.load_state_dict(saved["state_dict"], strict=True)
    model.to(DEVICE).eval().requires_grad_(False)

    blends = sorted({rule["blend"] for rule in rules.values()})
    seen_scores, seen_heads, seen_margin = predict_official(model, pairs, seen_names, "seen", semantic, blends)
    unseen_scores, unseen_heads, unseen_margin = predict_official(model, pairs, unseen_names, "unseen", semantic, blends)
    boundary = len(train_pairs)
    membership = torch.arange(len(pairs)) < boundary
    perturbed = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for name, transform in PERTURBATIONS.items():
            values, _, margin = predict_official(model, pairs, seen_names, name, semantic, blends, transform)
            perturbed[name] = (values, margin)
    # Materialize every prediction for both frozen rules before loading per-image targets.
    predictions = {}
    for name, rule in rules.items():
        blend, gamma, beta = rule["blend"], rule["gamma"], rule["adaptive"]
        predictions[name] = {
            "seen": top_predictions(apply_rule(seen_scores[blend], membership, seen_margin, gamma, beta)),
            "unseen": top_predictions(apply_rule(unseen_scores[blend], membership, unseen_margin, gamma, beta)),
            "seen_only": top_predictions(seen_scores[blend][:, :boundary]),
            "unseen_only": top_predictions(unseen_scores[blend][:, boundary:], offset=boundary),
            "robust": {key: apply_rule(scores[blend], membership, margin, gamma, beta).topk(1, dim=1).indices
                       for key, (scores, margin) in perturbed.items()},
        }
    return predictions, unseen_scores, seen_heads, unseen_heads, unseen_margin


def score_trial(run, pairs, forward_pass, y_seen, y_unseen):
    predictions, unseen_scores, seen_heads, unseen_heads, unseen_margin = forward_pass
    rules = run["prediction_rules"]
    boundary = len(train_pairs)
    y_all = torch.cat((y_seen, y_unseen))

    reports, ranking, per_pair, robustness, restricted = [], [], [], [], []
    same_crop_candidates = pairs[:, 0].eq(pairs[y_unseen, :1])
    for name, prediction in predictions.items():
        rule = rules[name]
        seen_head_hits = seen_heads[rule["blend"]].eq(pairs[y_seen])
        unseen_head_hits = unseen_heads[rule["blend"]].eq(pairs[y_unseen])
        head_top1 = {
            f"{split}_{head}_evidence_top1": round(100 * matches[:, col].float().mean().item(), 2)
            for split, matches in (("seen", seen_head_hits), ("unseen", unseen_head_hits))
            for col, head in enumerate(("crop", "disease"))
        }
        raw_scores = unseen_scores[rule["blend"]]
        crop_oracle = raw_scores.masked_fill(~same_crop_candidates, -torch.inf).topk(1, dim=1).indices
        raw_margins = raw_scores.gather(1, y_unseen[:, None]).squeeze(1) - raw_scores[:, :boundary].amax(dim=1)
        penalty = (rule["gamma"] + rule["adaptive"] * unseen_margin).clamp_min(0)
        final_margins = raw_margins + penalty
        only_correct = prediction["unseen_only"][1].squeeze(1).eq(y_unseen)
        metrics = {"model": name, "recipe": run["recipe"], **rule, "epoch": run["selection"]["epoch"],
                   "candidates": len(pairs), **head_top1,
                   "raw_true_crop_candidate_top1": prediction_accuracy(crop_oracle, y_unseen)}
        for k in (1, 5):
            metrics[f"seen_top{k}"] = prediction_accuracy(prediction["seen"][k], y_seen)
            metrics[f"unseen_top{k}"] = prediction_accuracy(prediction["unseen"][k], y_unseen)
            joined = torch.cat((prediction["seen"][k], prediction["unseen"][k]))
            metrics[f"overall_top{k}"] = prediction_accuracy(joined, y_all)
        if name.endswith("_raw"):
            metrics["historical_B0_unseen_top1"] = B0_UNSEEN_REFERENCE
            metrics["relative_unseen_gain_vs_B0_pct"] = round(
                100 * (metrics["unseen_top1"] / B0_UNSEEN_REFERENCE - 1), 2)
            metrics["meets_30pct_relative_target"] = metrics["unseen_top1"] >= RAW_UNSEEN_TARGET
            for target in (40, 50):
                metrics[f"meets_unseen_{target}pct"] = metrics["unseen_top1"] >= target
        metrics["harmonic_mean"] = harmonic_mean(metrics["seen_top1"], metrics["unseen_top1"])
        metrics["abs_seen_unseen_gap_pp"] = round(abs(metrics["seen_top1"] - metrics["unseen_top1"]), 2)
        seen_winner, unseen_winner = (prediction[split][1].squeeze(1) for split in ("seen", "unseen"))
        metrics["seen_predicted_unseen"] = int((seen_winner >= boundary).sum())
        for split, winner, target in (("seen", seen_winner, y_seen), ("unseen", unseen_winner, y_unseen)):
            concept_hits = pairs[winner].eq(pairs[target]).float().mean(dim=0) * 100
            metrics[f"{split}_test_crop_acc"], metrics[f"{split}_test_disease_acc"] = [
                round(float(value), 2) for value in concept_hits]
        reports.append(metrics)

        correct = unseen_winner.eq(y_unseen)
        predicts_seen = unseen_winner < boundary
        blocked = only_correct & predicts_seen
        same_crop = pairs[unseen_winner, 0].eq(pairs[y_unseen, 0])
        ranking.append({"model": name, "unseen_images": len(y_unseen), "joint_correct": int(correct.sum()),
                        "unseen_only_correct": int(only_correct.sum()), "unseen_predicted_seen": int(predicts_seen.sum()),
                        "blocked_by_seen": int(blocked.sum()), "blocked_same_crop": int((blocked & same_crop).sum()),
                        "blocked_other_crop": int((blocked & ~same_crop).sum()),
                        "median_true_pair_minus_top_seen_score": float(final_margins.median())})
        for index in range(boundary, len(pairs)):
            mask = y_unseen.eq(index)
            crop, disease = pairs[index].tolist()
            per_pair.append({"model": name, "crop": crop, "disease": disease, "support": int(mask.sum()),
                             "joint_correct": int((mask & correct).sum()),
                             "unseen_only_correct": int((mask & only_correct).sum()),
                             "blocked_by_seen": int((mask & blocked).sum())})
        for key, predictions_at_one in prediction["robust"].items():
            accuracy = prediction_accuracy(predictions_at_one, y_seen)
            robustness.append({"model": name, "condition": key, "seen_top1": accuracy,
                               "drop_pp": round(metrics["seen_top1"] - accuracy, 2)})
        for split, targets, count in (("seen_only", y_seen, boundary),
                ("unseen_only", y_unseen, len(pairs) - boundary)):
            restricted.append({"model": name, "split": split, "candidates": count,
                               **{f"top{k}": prediction_accuracy(prediction[split][k], targets) for k in (1, 5)}})
    return ({"results": reports, "restricted_candidates": restricted, "robustness": robustness,
             "unseen_ranking_diagnostics": ranking, "unseen_per_pair": per_pair},
            {name: pred["seen"][1].squeeze(1) for name, pred in predictions.items()})


def official_evaluation():
    if DEBUG or not runs or any(read_final_lock(trial) != runs[trial.code] for trial in trials):
        raise RuntimeError("Complete all full refits and lock their settings before official evaluation.")
    if any(runs[trial.code]["research_plan"] != plan for trial in trials):
        raise ValueError("Checkpoints refer to different research plans.")
    if sha256_file(SEMANTIC_CHECKPOINT) != plan["clip_sha256"]:
        raise RuntimeError("CLIP weights or concept text bank changed after selection.")
    saved_clip = torch.load(SEMANTIC_CHECKPOINT, map_location="cpu", weights_only=True)
    clip_model = open_clip.create_model(CLIP_MODEL, pretrained=None)
    clip_model.visual.load_state_dict(saved_clip["visual"], strict=True)
    semantic = (clip_model.to(DEVICE).eval(), {key: val.to(DEVICE) for key, val in saved_clip["bank"].items()})
    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    seen_names, unseen_names = map(official_filenames, paths)
    if set(seen_names) & set(unseen_names) or set(train_full.image) & (set(seen_names) | set(unseen_names)):
        raise ValueError("Image names overlap between training and evaluation splits.")
    forward_passes = {trial.code: predict_trial(trial, runs[trial.code], pairs, seen_names, unseen_names, semantic if trial.code == "A2" else None)
                      for trial in trials}
    # Every frozen checkpoint and rule has predicted before labels are joined.
    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], seen_names, rows, seen=True)
    y_unseen = official_targets(paths[1], unseen_names, rows, seen=False)
    summary = {"selection": runs, "samples": {"seen": len(y_seen), "unseen": len(y_unseen)},
               "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist()}
    winners = {}
    for code, predictions in forward_passes.items():
        report, seen_winners = score_trial(runs[code], pairs, predictions, y_seen, y_unseen)
        winners.update(seen_winners)
        for key, value in report.items():
            for row in value:
                row.setdefault("recipe", code)
            summary.setdefault(key, []).extend(value)
    summary["joint_gains"] = comparisons(summary["results"], OFFICIAL_METRICS)
    return summary, winners, y_seen


evaluation = None if DEBUG else official_evaluation()
if evaluation is None:
    print("Debug logs:", trials[0].development_dir)


# 6. Save

Open `joint_gains.csv` to compare Fusion raw and Balanced with their matching A2 and B0 controls. `all_three_and_balance` requires three positive gains, a gap within 10 points and a seen loss within 2 points relative to that row's control. The development folder includes the independent proxy check. The official folder includes separate raw and calibrated metrics, restricted results, robustness, ranking diagnostics and confusion matrices. Run folders are unique.

- `results/approach2_a2_clip_balance/<run_id>/{development,final,official}/`
- `checkpoints/approach2_a2_clip_balance/<run_id>/{B0/model.pt,A2/model.pt,clip_semantics.pt}`

In [ ]:
def save_results(summary, seen_winners, seen_targets):
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    # Align the team confusion plot order only after predictions and metrics are fixed.
    pairs = [tuple(pair) for pair in summary["candidate_pairs"]]
    lookup = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if set(pairs) != lookup.keys():
        raise ValueError("Candidate identities differ from the common evaluator.")
    to_common = torch.tensor([lookup[pair] for pair in pairs])
    canonical_targets = to_common[seen_targets]
    if (canonical_targets >= len(train_pairs)).any():
        raise ValueError("The common confusion plot expects seen target rows.")
    for name, winners in seen_winners.items():
        votes = F.one_hot(to_common[winners], num_classes=len(pairs)).float()
        stem = (OFFICIAL_DIR / name).relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(votes, canonical_targets, stem, final_evaluation=True)

    table = pd.DataFrame(summary["results"])
    table.to_csv(OFFICIAL_DIR / f"{MODEL_NAME}_comparison.csv", index=False)
    for name in ("joint_gains", "restricted_candidates", "robustness", "unseen_ranking_diagnostics", "unseen_per_pair"):
        pd.DataFrame(summary[name]).to_csv(OFFICIAL_DIR / f"{name}.csv", index=False)
    for row in summary["results"]:
        atomic_json(OFFICIAL_DIR / f"{row['model']}.json", row)
    atomic_json(OFFICIAL_DIR / f"{MODEL_NAME}_comparison.json", summary)
    return table


if evaluation is not None:
    table = save_results(*evaluation)
    display(table[["model", "recipe", "blend", "gamma", "adaptive", "seen_top1", "unseen_top1", "unseen_top5",
                   "overall_top1", "harmonic_mean", "unseen_disease_evidence_top1", "seen_predicted_unseen"]])
    display(pd.DataFrame(evaluation[0]["joint_gains"]))
    for trial in trials:
        print(trial.code, "checkpoint:", trial.checkpoint.resolve())
    print("CLIP checkpoint:", SEMANTIC_CHECKPOINT.resolve())
    print("Results:", OFFICIAL_DIR.resolve())
